# Atlas Post-Processing Comparison

Compares four conditions for one model:
1. **Raw** — nnU-Net predictions as-is
2. **Random Atlas** — random atlas, perturbed (±10° / ±5 mm), no anatomy rules
3. **Disease Atlas** — disease-matched atlas, unperturbed, no anatomy rules
4. **Disease Atlas + Rules** — disease-matched atlas + disease-aware anatomy priors + adjacency correction

Outputs are saved to `atlas_postprocessed/` and Dice CSVs are cached so re-running the notebook skips already-processed files.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME      = "nnUNetTrainerDA5Curriculum_1000epochs__nnUNetResEncUNetMPlans__3d_fullres"
FOLDS           = ["fold_0"]   # extend to ["fold_0","fold_1",...,"fold_4"] for all folds
SEED            = 42
FORCE_RECOMPUTE = False    # set False to skip already-processed files

# ── Cache control ──────────────────────────────────────────────────────────────
# Set True to delete all cached atlas outputs and rerun from scratch.
# Useful when the pipeline code changes and cached outputs are stale.
CLEAR_CACHE = False

# ── Debug toggle ───────────────────────────────────────────────────────────────
# Set True to run only the first prediction case (fast sanity check).
# Set False to run the full fold.
DEBUG_SINGLE_CASE = False

In [2]:
import sys
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT        = Path("/Users/saschastocker/Documents/Stanford/Akshay/DataInvestigation")
PRED_ROOT   = ROOT / "predictions"

# Pre-built disease-stratified atlas library (one deformed atlas per disease group).
GT_FOLDER   = ROOT / "atlases"

# Ground-truth labels for the test cases — local copy inside DataInvestigation
# so Jupyter can read them (macOS TCC blocks /Documents/data/ from Jupyter).
GT_LABELS_TS = ROOT / "gt_labels_ts"

DISEASE_MAP = ROOT / "disease_map.json"
OUT_ROOT    = ROOT / "atlas_postprocessed"

POSTPROC_LIB = Path("/Users/saschastocker/Documents/Stanford/Akshay/nnunet_chd_postprocessing")
sys.path.insert(0, str(POSTPROC_LIB))

for p in [GT_FOLDER, GT_LABELS_TS, DISEASE_MAP, POSTPROC_LIB, PRED_ROOT / MODEL_NAME]:
    assert p.exists(), f"Missing: {p}"
print("All paths OK")
print(f"Atlas library : {GT_FOLDER} ({len(list(GT_FOLDER.glob('*.nii.gz')))} atlases)")
print(f"GT labels (Ts): {GT_LABELS_TS} ({len(list(GT_LABELS_TS.glob('*.nii.gz')))} files)")

All paths OK
Atlas library : /Users/saschastocker/Documents/Stanford/Akshay/DataInvestigation/atlases (17 atlases)
GT labels (Ts): /Users/saschastocker/Documents/Stanford/Akshay/DataInvestigation/gt_labels_ts (14 files)


In [3]:
# ── Dice helpers ───────────────────────────────────────────────────────────────
# Names match dataset.json exactly; WH = whole-heart (all foreground combined)
CLASSES   = ['WH', 'LV-BP', 'RV-BP', 'LA', 'RA', 'Myo', 'Aorta', 'Pulmonary']
LABEL_MAP = {'WH': None, 'LV-BP': 1, 'RV-BP': 2, 'LA': 3, 'RA': 4, 'Myo': 5, 'Aorta': 6, 'Pulmonary': 7}

def dice_score(pred, gt, label):
    if label is None:
        p, g = (pred > 0), (gt > 0)
    else:
        p, g = (pred == label), (gt == label)
    inter = np.sum(p & g)
    denom = np.sum(p) + np.sum(g)
    return float(2 * inter / denom) if denom > 0 else float('nan')

def find_gt(case_id):
    """Find GT file for a prediction case_id like 'ct_1004'.

    Searches GT_LABELS_TS for both plain (ct_XXXX.nii.gz) and
    _image-suffixed (ct_XXXX_image.nii.gz) variants.
    """
    for suffix in [".nii.gz", ".nii"]:
        for name in [f"{case_id}{suffix}", f"{case_id}_image{suffix}"]:
            p = GT_LABELS_TS / name
            if p.exists():
                return p
    return None

# Quick sanity check
_sample = list(GT_LABELS_TS.glob("*.nii.gz"))[:1]
if _sample:
    print(f"GT sample: {_sample[0].name}  →  find_gt returns: {find_gt(_sample[0].stem.replace('.nii',''))}")
else:
    print("WARNING: GT_LABELS_TS appears empty")

GT sample: ct_1064.nii.gz  →  find_gt returns: /Users/saschastocker/Documents/Stanford/Akshay/DataInvestigation/gt_labels_ts/ct_1064.nii.gz


In [ ]:
# ── Sanity Checks ─────────────────────────────────────────────────────────────
# Runs 5 diagnostic checks and writes output to ROOT/sanity_check.log.
# Check 4 (full pipeline) runs on MAX_PIPELINE_CASES only — the pipeline is slow.
# All other checks run on all cases (they are fast: NIfTI reads + numpy only).

import datetime
import traceback as _tb
from scipy.ndimage import label as nd_label

from chd_postprocessing.atlas import AtlasLibrary
from chd_postprocessing.atlas_pipeline import run_atlas_pipeline
from chd_postprocessing.anatomy_priors import correct_ao_pa_fragments
from chd_postprocessing.evaluate import dice_per_class
from chd_postprocessing.registration import register_atlas_to_pred
from chd_postprocessing.io_utils import (
    load_disease_map, get_disease_vec, get_voxel_spacing, resolve_case_id,
)
from chd_postprocessing.config import DISEASE_ANATOMY_RULES, DISEASE_FLAGS, LABELS, LABEL_NAMES

MAX_PIPELINE_CASES = 1   # Check 4 only — full pipeline is slow; 1 case is enough to diagnose

# ── Dual-output logger ────────────────────────────────────────────────────────
_sc_log_path = ROOT / "sanity_check.log"
_sc_fh = open(_sc_log_path, "w")

def log(msg=""):
    print(msg)
    _sc_fh.write(str(msg) + "\n")
    _sc_fh.flush()

# ── Gather cases ───────────────────────────────────────────────────────────────
_sc_fold        = FOLDS[0]
_sc_pred_folder = PRED_ROOT / MODEL_NAME / "imagesTs" / _sc_fold
_sc_pred_files  = sorted(_sc_pred_folder.glob("*.nii.gz"))
if DEBUG_SINGLE_CASE:
    _sc_pred_files = _sc_pred_files[:1]

log(f"{'='*70}")
log(f"SANITY CHECK — {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log(f"Model : {MODEL_NAME}")
log(f"Fold  : {_sc_fold}")
log(f"Cases : {len(_sc_pred_files)}  ({[p.name for p in _sc_pred_files[:3]]}{'...' if len(_sc_pred_files)>3 else ''})")
log(f"{'='*70}")

# ── Pre-load atlas library (shared across all cases) ──────────────────────────
try:
    _sc_library     = AtlasLibrary.load_all(GT_FOLDER, disease_map_path=DISEASE_MAP)
    _sc_disease_map = load_disease_map(DISEASE_MAP)
    log(f"\nAtlas library: {len(_sc_library)} entries")
except Exception as _e:
    log(f"[FATAL] Could not load atlas library: {_e}")
    _sc_library     = None
    _sc_disease_map = {}

# ─────────────────────────────────────────────────────────────────────────────
# CHECK 5: Stale cache in OUT_ROOT  (run once, before per-case loop)
# ─────────────────────────────────────────────────────────────────────────────
log(f"\n{'─'*60}")
log(f"CHECK 5: Stale cache in OUT_ROOT")
log(f"{'─'*60}")
try:
    _cached_files = list(OUT_ROOT.rglob("*.nii.gz"))
    if not _cached_files:
        log(f"  No cached NIfTI files — cache is clean.")
    else:
        _mtimes = [p.stat().st_mtime for p in _cached_files]
        _oldest = datetime.datetime.fromtimestamp(min(_mtimes)).strftime("%Y-%m-%d %H:%M:%S")
        _newest = datetime.datetime.fromtimestamp(max(_mtimes)).strftime("%Y-%m-%d %H:%M:%S")
        log(f"  {len(_cached_files)} cached NIfTI files in {OUT_ROOT}")
        log(f"  Oldest: {_oldest}   Newest: {_newest}")
        if not FORCE_RECOMPUTE:
            log(f"  ⚠ FORCE_RECOMPUTE=False — notebook will load these files, not rerun pipeline")
        for _p in sorted(_cached_files)[:5]:
            log(f"    {_p.relative_to(OUT_ROOT)}")
        if len(_cached_files) > 5:
            log(f"    … and {len(_cached_files)-5} more")
except Exception as _e:
    log(f"  [ERROR] {_e}")

# ─────────────────────────────────────────────────────────────────────────────
# Per-case checks: 1, 2, 3  (fast — NIfTI reads + numpy only)
# ─────────────────────────────────────────────────────────────────────────────
for _sc_idx, _sc_pred_path in enumerate(_sc_pred_files):
    _sc_case_id = resolve_case_id(_sc_pred_path.name)
    _sc_gt_path = find_gt(_sc_case_id)

    log(f"\n{'#'*70}")
    log(f"CASE {_sc_idx+1}/{len(_sc_pred_files)}: {_sc_case_id}")
    log(f"  Pred: {_sc_pred_path}")
    log(f"  GT  : {_sc_gt_path}")
    log(f"{'#'*70}")

    try:
        _sc_pred_img = nib.load(str(_sc_pred_path))
        _sc_pred_arr = np.asarray(_sc_pred_img.dataobj).astype(np.int32)
        _sc_spacing  = get_voxel_spacing(_sc_pred_img.header)
        _sc_gt_arr   = (np.asarray(nib.load(str(_sc_gt_path)).dataobj).astype(np.int32)
                        if _sc_gt_path else None)
    except Exception as _e:
        log(f"  [ERROR] Could not load: {_e}")
        continue

    _sc_disease_vec     = get_disease_vec(_sc_disease_map, _sc_case_id) or [0]*8
    _sc_active_diseases = [DISEASE_FLAGS[i] for i, f in enumerate(_sc_disease_vec) if f]
    log(f"  Disease: {_sc_active_diseases if _sc_active_diseases else ['Normal']}")

    # ── CHECK 1: Label structure ───────────────────────────────────────────────
    log(f"\n{'─'*60}")
    log(f"CHECK 1: Label structure (voxel counts + connected components)")
    log(f"{'─'*60}")
    try:
        log(f"\n  {'Lbl':<4} {'Name':<12} {'Pred vox':>10} {'Pred CCs':>9} "
            f"{'GT vox':>10} {'GT CCs':>9}")
        log(f"  {'-'*58}")
        _sc_multi_cc = []
        for _lbl in range(1, 8):
            _name   = LABEL_NAMES.get(_lbl, str(_lbl))
            _pmask  = _sc_pred_arr == _lbl
            _pcount = int(_pmask.sum())
            _, _pn  = nd_label(_pmask)
            _gcount, _gn = (0, 0)
            if _sc_gt_arr is not None:
                _gmask  = _sc_gt_arr == _lbl
                _gcount = int(_gmask.sum())
                _, _gn  = nd_label(_gmask)
            _flag = "  ← ⚠ multi-CC" if _pn > 1 else ""
            log(f"  {_lbl:<4} {_name:<12} {_pcount:>10,} {_pn:>9} {_gcount:>10,} {_gn:>9}{_flag}")
            if _pn > 1:
                _sc_multi_cc.append(_lbl)
                _cc_vol, _ = nd_label(_pmask)
                _sizes = sorted([int((_cc_vol==c).sum()) for c in range(1, _pn+1)], reverse=True)
                log(f"       CC sizes: [{', '.join(f'{s:,}' for s in _sizes[:6])}]")

        if not _sc_multi_cc:
            log(f"\n  NOTE: No label has >1 CC — component-level correction cannot help.")
        else:
            log(f"\n  Labels with >1 CC: {[LABEL_NAMES[l] for l in _sc_multi_cc]}")

        if _sc_gt_arr is not None:
            log(f"\n  Confusion (pred label vs GT, mis-assigned voxels > 100):")
            _found = False
            for _pl in range(1, 8):
                _pmask = _sc_pred_arr == _pl
                if not _pmask.any(): continue
                for _gl in range(0, 8):
                    if _gl == _pl: continue
                    _ov = int(np.sum(_pmask & (_sc_gt_arr == _gl)))
                    if _ov > 100:
                        _pn_s = LABEL_NAMES.get(_pl, str(_pl))
                        _gn_s = LABEL_NAMES.get(_gl, str(_gl)) if _gl > 0 else "background"
                        log(f"    Pred={_pn_s:<10} should_be={_gn_s:<12}: {_ov:>8,} voxels")
                        _found = True
            if not _found:
                log(f"    (No mis-assigned blocks > 100 voxels)")
    except Exception as _e:
        log(f"  [ERROR] Check 1: {_e}"); log(_tb.format_exc())

    # ── CHECK 2: Registration quality (centroid only — per-structure is too slow) ──
    log(f"\n{'─'*60}")
    log(f"CHECK 2: Registration quality — IoC after centroid alignment")
    log(f"{'─'*60}")
    try:
        if _sc_library is None:
            raise RuntimeError("Atlas library not loaded")

        _sc_rng        = random.Random(SEED)
        _sc_rand_entry = _sc_library.select_for_case(_sc_disease_vec, _sc_rng, mode="random",
                                                      exclude_case_ids=[_sc_case_id])
        _sc_dis_entry  = _sc_library.select_for_case(_sc_disease_vec, _sc_rng, mode="best_match",
                                                      exclude_case_ids=[_sc_case_id])
        _sc_rand_entry.load()
        _sc_dis_entry.load()

        log(f"\n  Random  atlas: {_sc_rand_entry.case_id} ({_sc_rand_entry.disease_name})")
        log(f"  Disease atlas: {_sc_dis_entry.case_id} "
            f"({_sc_dis_entry.disease_name}, hamming={_sc_dis_entry.hamming_distance(_sc_disease_vec)})")

        for _atlas_tag, _atlas_arr in [("RANDOM",  _sc_rand_entry.labels),
                                        ("DISEASE", _sc_dis_entry.labels)]:
            _reg = register_atlas_to_pred(_atlas_arr, _sc_pred_arr, _sc_spacing, mode="centroid")
            log(f"\n  === {_atlas_tag} atlas — centroid registration ===")
            log(f"  {'Lbl':<4} {'Name':<12} {'Pred vox':>10} {'Atlas vox':>10} "
                f"{'Intersect':>10} {'IoC':>7}")
            log(f"  {'-'*58}")
            _all_low = True
            for _lbl in range(1, 8):
                _name  = LABEL_NAMES.get(_lbl, str(_lbl))
                _pmask = _sc_pred_arr == _lbl
                _amask = _reg == _lbl
                _pcnt  = int(_pmask.sum())
                _acnt  = int(_amask.sum())
                _inter = int(np.sum(_pmask & _amask))
                _ioc   = _inter / _pcnt if _pcnt > 0 else float("nan")
                _ioc_s = f"{_ioc:.3f}" if not np.isnan(_ioc) else "  NaN"
                _warn  = "  ← LOW" if (not np.isnan(_ioc) and _ioc < 0.10) else ""
                if not np.isnan(_ioc) and _ioc >= 0.10:
                    _all_low = False
                log(f"  {_lbl:<4} {_name:<12} {_pcnt:>10,} {_acnt:>10,} {_inter:>10,} "
                    f"{_ioc_s:>7}{_warn}")
            if _all_low:
                log(f"\n  ⚠ ALL IoC < 0.10 — atlas useless after centroid registration.")
    except Exception as _e:
        log(f"  [ERROR] Check 2: {_e}"); log(_tb.format_exc())

    # ── CHECK 3: Anatomy priors + brute-force AO↔PA swap ─────────────────────
    log(f"\n{'─'*60}")
    log(f"CHECK 3: Anatomy priors (correct_ao_pa_fragments + global AO↔PA swap)")
    log(f"{'─'*60}")
    try:
        _sc_corr_arr, _sc_flog = correct_ao_pa_fragments(
            _sc_pred_arr.copy(), _sc_disease_vec, _sc_spacing)
        _sc_n_reassigned = len(_sc_flog.get("reassigned", []))

        if _sc_flog.get("skipped_disease", False):
            log(f"\n  SKIPPED by disease rule.")
        elif _sc_n_reassigned == 0:
            log(f"\n  correct_ao_pa_fragments: 0 fragments reassigned.")
        else:
            log(f"\n  correct_ao_pa_fragments: {_sc_n_reassigned} fragment(s) reassigned:")
            for _entry in _sc_flog["reassigned"]:
                _on = LABEL_NAMES.get(_entry["original_label"], str(_entry["original_label"]))
                _nn = LABEL_NAMES.get(_entry["assigned_label"],  str(_entry["assigned_label"]))
                log(f"    {_on} → {_nn}: {_entry['size']:,} vox "
                    f"(correct={_entry['correct_score']:.3f}, wrong={_entry['wrong_score']:.3f})")

        if _sc_gt_arr is not None:
            _ao_lbl, _pa_lbl = LABELS["AO"], LABELS["PA"]
            _d_raw  = dice_per_class(_sc_pred_arr,  _sc_gt_arr, list(range(1, 8)))
            _d_frag = dice_per_class(_sc_corr_arr,  _sc_gt_arr, list(range(1, 8)))
            _ao_b, _ao_a = _d_raw[_ao_lbl], _d_frag[_ao_lbl]
            _pa_b, _pa_a = _d_raw[_pa_lbl], _d_frag[_pa_lbl]
            log(f"\n  Dice after correct_ao_pa_fragments:")
            log(f"    AO: {_ao_b:.4f} → {_ao_a:.4f}  (Δ {_ao_a-_ao_b:+.4f})")
            log(f"    PA: {_pa_b:.4f} → {_pa_a:.4f}  (Δ {_pa_a-_pa_b:+.4f})")

            _swapped = _sc_pred_arr.copy()
            _swapped[_sc_pred_arr == _ao_lbl] = _pa_lbl
            _swapped[_sc_pred_arr == _pa_lbl] = _ao_lbl
            _d_swap = dice_per_class(_swapped, _sc_gt_arr, list(range(1, 8)))
            _ao_sw, _pa_sw = _d_swap[_ao_lbl], _d_swap[_pa_lbl]
            log(f"\n  Brute-force global AO↔PA swap:")
            log(f"    AO: {_ao_b:.4f} → {_ao_sw:.4f}  (Δ {_ao_sw-_ao_b:+.4f})")
            log(f"    PA: {_pa_b:.4f} → {_pa_sw:.4f}  (Δ {_pa_sw-_pa_b:+.4f})")
            if _ao_sw > _ao_b and _pa_sw > _pa_b:
                log(f"  ⚠ Swap improves BOTH — entire AO/PA globally mislabeled!")
            else:
                log(f"  Swap does not improve both — vessel sides appear correct.")
    except Exception as _e:
        log(f"  [ERROR] Check 3: {_e}"); log(_tb.format_exc())

# ─────────────────────────────────────────────────────────────────────────────
# CHECK 4: Full pipeline on first MAX_PIPELINE_CASES cases only (slow)
# ─────────────────────────────────────────────────────────────────────────────
log(f"\n{'─'*60}")
log(f"CHECK 4: Full pipeline — all 3 modes  (first {MAX_PIPELINE_CASES} case(s) only)")
log(f"{'─'*60}")
try:
    for _sc_pred_path in _sc_pred_files[:MAX_PIPELINE_CASES]:
        _sc_case_id = resolve_case_id(_sc_pred_path.name)
        _sc_gt_path = find_gt(_sc_case_id)
        log(f"\n  Case: {_sc_case_id}")
        for _mode in ["random_atlas", "disease_atlas", "disease_atlas_rules"]:
            log(f"\n  ─── {_mode} ───")
            try:
                _res = run_atlas_pipeline(
                    pred_path        = _sc_pred_path,
                    gt_folder        = GT_FOLDER,
                    output_path      = None,
                    disease_map_path = DISEASE_MAP,
                    gt_path          = _sc_gt_path,
                    mode             = _mode,
                    seed             = SEED,
                    registration_mode= "per_structure",
                )
                _adj_n = len(_res.adjacency_log or [])
                _br_n  = sum(e.get("a_to_b",0)+e.get("b_to_a",0) for e in (_res.boundary_log or []))
                log(f"  Atlas: {_res.atlas_case_id} ({_res.atlas_disease_name})")
                log(f"  was_relabeled: {_res.was_relabeled}   adj={_adj_n}   boundary={_br_n} vox")
                log(f"  {_res.reassignment_summary.splitlines()[0] if _res.reassignment_summary else ''}")
                if _res.dice_before is not None and _res.dice_after is not None:
                    log(f"\n  {'Class':<12} {'Before':>8} {'After':>8} {'Delta':>8}")
                    log(f"  {'-'*40}")
                    for _cls in _res.dice_before:
                        _b, _a = _res.dice_before[_cls], _res.dice_after[_cls]
                        if np.isnan(_b) or np.isnan(_a):
                            log(f"  {_cls:<12} {'NaN':>8} {'NaN':>8}")
                        else:
                            _d = _a - _b
                            _arrow = "  ↑" if _d > 0.005 else ("  ↓" if _d < -0.005 else "")
                            log(f"  {_cls:<12} {_b:>8.4f} {_a:>8.4f} {_d:>+8.4f}{_arrow}")
            except Exception as _me:
                log(f"  [ERROR] {_mode}: {_me}"); log(_tb.format_exc())
except Exception as _e:
    log(f"  [ERROR] Check 4: {_e}"); log(_tb.format_exc())

# ── Done ──────────────────────────────────────────────────────────────────────
log(f"\n{'='*70}")
log(f"Sanity check complete — {len(_sc_pred_files)} case(s).")
log(f"{'='*70}")
_sc_fh.close()
print(f"\nLog saved to: {_sc_log_path}")

In [ ]:
# ── Atlas pipeline ────────────────────────────────────────────────────────────
# Three modes (auto-set flags shown):
#
#   random_atlas        → perturbation=True,  anatomy=False, adjacency=False
#                         random atlas, ±10°/±5 mm deformation applied
#
#   disease_atlas       → perturbation=False, anatomy=False, adjacency=False
#                         disease-matched atlas used as-is; no rule corrections
#
#   disease_atlas_rules → perturbation=False, anatomy=True,  adjacency=True
#                         disease-matched atlas + disease-aware anatomy priors
#                         + adjacency-graph correction (runs after IoC step)
#
# Backward-compatible aliases still accepted:
#   "baseline"         → "random_atlas"
#   "disease_specific" → "disease_atlas_rules"

from chd_postprocessing.atlas_pipeline import run_atlas_pipeline
from chd_postprocessing.io_utils import resolve_case_id

print("run_atlas_pipeline ready.")

In [ ]:
# ── Run atlas post-processing ──────────────────────────────────────────────────
# Outputs:
#   atlas_postprocessed/random_atlas/<model>/<fold>/ct_XXXX.nii.gz
#   atlas_postprocessed/disease_atlas/<model>/<fold>/ct_XXXX.nii.gz
#   atlas_postprocessed/disease_atlas_rules/<model>/<fold>/ct_XXXX.nii.gz

# Clear cached atlas outputs if requested
if CLEAR_CACHE:
    import shutil
    for subdir in ["random_atlas", "disease_atlas", "disease_atlas_rules", "baseline", "disease"]:
        p = OUT_ROOT / subdir
        if p.exists():
            shutil.rmtree(p)
            print(f"Cleared cache: {p}")

ATLAS_CONDITIONS = [
    ("random_atlas",        "random_atlas",        "random_atlas"),
    ("disease_atlas",       "disease_atlas",       "disease_atlas"),
    ("disease_atlas_rules", "disease_atlas_rules", "disease_atlas_rules"),
]

atlas_log = []

for fold in FOLDS:
    pred_folder = PRED_ROOT / MODEL_NAME / "imagesTs" / fold
    pred_files  = sorted(pred_folder.glob("*.nii.gz"))
    if not pred_files:
        print(f"[{fold}] no predictions — skipping")
        continue

    if DEBUG_SINGLE_CASE:
        pred_files = pred_files[:1]
        print(f"\n{fold}  [DEBUG: 1 case only — {pred_files[0].name}]")
    else:
        print(f"\n{fold}  ({len(pred_files)} cases)")

    for i, pred_path in enumerate(tqdm(pred_files, desc=fold)):
        case_id = resolve_case_id(pred_path.name)

        for mode, out_subdir, mode_label in ATLAS_CONDITIONS:
            out_path = OUT_ROOT / out_subdir / MODEL_NAME / fold / pred_path.name
            if out_path.exists() and not FORCE_RECOMPUTE:
                atlas_log.append({"case_id": case_id, "mode": mode_label,
                                  "fold": fold, "status": "cached"})
                continue

            try:
                result = run_atlas_pipeline(
                    pred_path         = pred_path,
                    gt_folder         = GT_FOLDER,
                    output_path       = out_path,
                    disease_map_path  = DISEASE_MAP,
                    mode              = mode,
                    seed              = SEED + i,
                    registration_mode = "per_structure",
                )
                atlas_log.append({
                    "case_id":       case_id,
                    "mode":          mode_label,
                    "fold":          fold,
                    "status":        "computed",
                    "atlas":         result.atlas_case_id,
                    "atlas_disease": result.atlas_disease_name,
                    "was_relabeled": result.was_relabeled,
                    "adj_changes":   len(result.adjacency_log) if result.adjacency_log else 0,
                    "br_changes":    sum(e.get("a_to_b", 0) + e.get("b_to_a", 0)
                                        for e in (result.boundary_log or [])),
                })
            except Exception as exc:
                import traceback
                atlas_log.append({"case_id": case_id, "mode": mode_label,
                                  "fold": fold, "status": "error", "error": str(exc)})
                print(f"  [ERROR] {case_id} {mode}: {exc}")
                traceback.print_exc()

log_df = pd.DataFrame(atlas_log)
print("\nProcessing summary:")
print(log_df.groupby(["fold", "mode", "status"]).size().to_string())
print()
display(log_df)

In [ ]:
# ── Compute Dice scores ────────────────────────────────────────────────────────

def dice_for_folder(pred_folder, case_ids=None, cache_csv=None):
    """Compute Dice for all (or a subset of) predictions in pred_folder.

    Parameters
    ----------
    case_ids : if provided, only compute Dice for these case IDs.
               Used by DEBUG_SINGLE_CASE to keep Dice in sync with the pipeline run.
    """
    pred_folder = Path(pred_folder)
    if cache_csv and Path(cache_csv).exists() and not FORCE_RECOMPUTE:
        try:
            cached = pd.read_csv(cache_csv)
            if all(c in cached.columns for c in CLASSES):
                if case_ids is not None:
                    cached = cached[cached["case_id"].isin(case_ids)]
                return cached
            print(f"  Cache outdated, recomputing: {cache_csv}")
        except (pd.errors.EmptyDataError, Exception):
            print(f"  Cache invalid/empty, recomputing: {cache_csv}")

    rows = []
    missing_gt = []
    pred_paths = sorted(pred_folder.glob("*.nii.gz"))
    for pred_path in pred_paths:
        stem = pred_path.stem.replace(".nii", "")
        if case_ids is not None and stem not in case_ids:
            continue
        gt_path = find_gt(stem)
        if gt_path is None:
            missing_gt.append(stem)
            continue

        try:
            pred_arr = np.asarray(nib.load(str(pred_path)).dataobj).astype(np.int32)
        except Exception as e:
            print(f"  [SKIP] Cannot load pred {pred_path.name}: {e}")
            continue
        try:
            gt_arr = np.asarray(nib.load(str(gt_path)).dataobj).astype(np.int32)
        except Exception as e:
            print(f"  [SKIP] Cannot load GT {gt_path}: {e}")
            continue

        row = {"case_id": stem}
        for cls in CLASSES:
            row[cls] = dice_score(pred_arr, gt_arr, LABEL_MAP[cls])
        rows.append(row)

    if missing_gt:
        print(f"  [WARN] No GT for: {missing_gt}")

    df = pd.DataFrame(rows, columns=["case_id"] + CLASSES) if rows else pd.DataFrame(columns=["case_id"] + CLASSES)
    # Only write cache when running full fold (not debug subset)
    if cache_csv and rows and case_ids is None:
        Path(cache_csv).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache_csv, index=False)
    return df


# Determine which case IDs to score (None = all)
# Include both "computed" and "cached" so DEBUG_SINGLE_CASE always limits to 1 case.
_debug_case_ids = None
if DEBUG_SINGLE_CASE and "log_df" in dir():
    _computed = log_df[log_df["status"].isin(["computed", "cached"])]["case_id"].unique().tolist()
    if _computed:
        _debug_case_ids = set(_computed)
        print(f"[DEBUG] Scoring only: {sorted(_debug_case_ids)}")

DICE_CONDITIONS = [
    ("Raw",                  PRED_ROOT / MODEL_NAME / "imagesTs"),
    ("Random Atlas",         OUT_ROOT / "random_atlas"        / MODEL_NAME),
    ("Disease Atlas",        OUT_ROOT / "disease_atlas"       / MODEL_NAME),
    ("Disease Atlas+Rules",  OUT_ROOT / "disease_atlas_rules" / MODEL_NAME),
]

all_rows = []

for fold in tqdm(FOLDS, desc="Dice"):
    for cond_label, base_folder in DICE_CONDITIONS:
        folder = Path(base_folder) / fold
        if not folder.exists():
            print(f"  [WARN] Folder missing: {folder}")
            continue
        df = dice_for_folder(folder, case_ids=_debug_case_ids,
                             cache_csv=None if DEBUG_SINGLE_CASE else ROOT / "dice_cache" / (cond_label.replace(" ","_").replace("+","p") + "_" + fold + ".csv"))
        if df.empty:
            print(f"  [WARN] No Dice rows for {cond_label} {fold}")
            continue
        df["condition"] = cond_label
        df["fold"]      = fold
        all_rows.append(df)

if not all_rows:
    raise RuntimeError("No Dice rows computed — check GT_LABELS_TS and that pipeline ran successfully.")

all_dice = pd.concat(all_rows, ignore_index=True)
print(f"\nTotal rows: {len(all_dice)}")
display(all_dice.groupby("condition")[CLASSES].mean().round(3))
# ── Disease annotation ────────────────────────────────────────────────────────
from chd_postprocessing.io_utils import load_disease_map, get_disease_vec
from chd_postprocessing.config import DISEASE_FLAGS

_dm = load_disease_map(DISEASE_MAP)

def get_disease_str(case_id):
    vec = get_disease_vec(_dm, case_id)
    if vec is None:
        return "Unknown"
    flags = [DISEASE_FLAGS[i] for i, f in enumerate(vec) if f]
    return "+".join(flags) if flags else "Normal"

all_dice["disease"] = all_dice["case_id"].apply(get_disease_str)
print(f"Disease groups: {sorted(all_dice['disease'].unique())}")

In [ ]:
# ── Mean Dice summary + Δ vs Raw ───────────────────────────────────────────────

cond_display_order = ["Raw", "Random Atlas", "Disease Atlas", "Disease Atlas+Rules"]
present_conds = [c for c in cond_display_order if c in all_dice["condition"].unique()]

summary = all_dice.groupby("condition")[CLASSES].mean().reindex(present_conds).round(3)
display(summary)

if "Raw" in summary.index:
    delta = summary.drop("Raw").subtract(summary.loc["Raw"])
    print("\nΔ vs Raw (positive = improvement):")
    display(delta.round(4).style.background_gradient(cmap='RdYlGn', vmin=-0.02, vmax=0.02))

# ── Per-disease stratified table ───────────────────────────────────────────────
if "disease" in all_dice.columns:
    print("\n" + "="*60)
    print("Per-disease-group mean Dice")
    print("="*60)
    for disease_group in sorted(all_dice["disease"].unique()):
        sub = all_dice[all_dice["disease"] == disease_group]
        n = len(sub[sub["condition"] == "Raw"])
        print(f"\n{disease_group}  (n={n} cases):")
        grp = sub.groupby("condition")[CLASSES].mean().reindex(present_conds).round(4)
        display(grp)
        if "Raw" in grp.index and len(grp) > 1:
            d = grp.drop("Raw").subtract(grp.loc["Raw"])
            display(d.round(4).style.background_gradient(cmap='RdYlGn', vmin=-0.05, vmax=0.05))

In [ ]:
# ── Per-class violin: Raw vs three atlas conditions ────────────────────────────

COND_ORDER  = ["Raw", "Random Atlas", "Disease Atlas", "Disease Atlas+Rules"]
COND_COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
palette     = dict(zip(COND_ORDER, COND_COLORS))

long_df = all_dice.melt(
    id_vars=["case_id", "condition", "fold"],
    value_vars=CLASSES,
    var_name="Structure",
    value_name="Dice",
)

fig, axes = plt.subplots(2, 4, figsize=(22, 10), sharey=False)
axes = axes.flatten()

short_labels = ["Raw", "Random\nAtlas", "Disease\nAtlas", "Disease\nAtlas+Rules"]

for ax, cls in zip(axes, CLASSES):
    sub     = long_df[long_df["Structure"] == cls].dropna(subset=["Dice"])
    present = [c for c in COND_ORDER if c in sub["condition"].unique()]
    short   = short_labels[:len(present)]

    sns.violinplot(
        data=sub, x="condition", y="Dice",
        order=present, palette=palette,
        inner=None, cut=0, linewidth=1.2, ax=ax
    )
    sns.stripplot(
        data=sub, x="condition", y="Dice",
        order=present, palette=palette,
        size=4, alpha=0.6, jitter=True, ax=ax
    )

    means = sub.groupby("condition")["Dice"].mean()
    for xi, c in enumerate(present):
        ax.scatter(xi, means[c], marker="D", color="black", s=50, zorder=6)

    ax.set_title(cls, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Dice" if ax is axes[0] or ax is axes[4] else "")
    ax.set_xticklabels(short, fontsize=8)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(axis="y", alpha=0.3)

    n = sub[sub["condition"] == "Raw"]["Dice"].notna().sum()
    ax.text(0.97, 0.02, f"n={n}", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=8, color="gray")

fig.suptitle(
    f"Atlas Post-processing: Dice Comparison\n{MODEL_NAME}\nFolds: {', '.join(FOLDS)}",
    fontsize=12, fontweight="bold", y=1.02
)

from matplotlib.patches import Patch
handles = [Patch(color=palette[c], label=c) for c in COND_ORDER]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.01, 1.0), fontsize=9)

plt.tight_layout()
out_fig = ROOT / "atlas_comparison_violin.png"
plt.savefig(out_fig, dpi=150, bbox_inches="tight")
print(f"Saved: {out_fig}")
plt.show()

In [ ]:
# ── Δ Dice per patient: did atlas correction help or hurt? ────────────────────

raw_pivot = all_dice[all_dice["condition"] == "Raw"].set_index("case_id")

comparison_conditions = [
    ("Random Atlas",        COND_COLORS[1]),
    ("Disease Atlas",       COND_COLORS[2]),
    ("Disease Atlas+Rules", COND_COLORS[3]),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 9), sharey=False)
axes = axes.flatten()

for ax, cls in zip(axes, CLASSES):
    for xi, (cond, color) in enumerate(comparison_conditions, start=1):
        cdf    = all_dice[all_dice["condition"] == cond].set_index("case_id")
        common = raw_pivot.index.intersection(cdf.index)
        if common.empty:
            continue
        delta = cdf.loc[common, cls] - raw_pivot.loc[common, cls]
        ax.scatter([xi] * len(delta), delta.values, color=color, alpha=0.65, s=35)
        ax.scatter(xi, delta.mean(), marker="D", color="black", s=55, zorder=6)

    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_title(cls, fontsize=12, fontweight="bold")
    ax.set_ylabel("Δ Dice vs Raw" if ax is axes[0] or ax is axes[4] else "")
    ax.set_xticks(range(1, len(comparison_conditions) + 1))
    ax.set_xticklabels(["Random\nAtlas", "Disease\nAtlas", "Disease\nAtlas+Rules"], fontsize=8)
    ax.set_xlim(0.4, len(comparison_conditions) + 0.6)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    f"Δ Dice (Atlas − Raw) per patient\n{MODEL_NAME}\nFolds: {', '.join(FOLDS)}",
    fontsize=12, fontweight="bold", y=1.02
)

from matplotlib.patches import Patch
handles = [Patch(color=c, label=l) for l, c in comparison_conditions]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.01, 1.0), fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / "atlas_delta.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Per-patient mean Dice table ────────────────────────────────────────────────

tmp = all_dice.copy()
tmp["mean_Dice"] = tmp[CLASSES].mean(axis=1)
pivot = tmp.pivot_table(index="case_id", columns="condition", values="mean_Dice").round(4)

if "Raw" in pivot.columns:
    for cond in ["Random Atlas", "Disease Atlas", "Disease Atlas+Rules"]:
        if cond in pivot.columns:
            short = cond.replace("Disease Atlas+Rules", "DA+Rules").replace("Disease Atlas", "DAtlas").replace("Random Atlas", "RAtlas")
            pivot[f"Δ {short}"] = (pivot[cond] - pivot["Raw"]).round(4)

delta_cols = [c for c in pivot.columns if c.startswith("Δ")]
if delta_cols:
    pivot = pivot.sort_values(delta_cols[-1], ascending=False)   # sort by best condition

display(pivot.style.background_gradient(
    subset=delta_cols, cmap="RdYlGn", vmin=-0.05, vmax=0.05
))